In [ ]:
pip install scikit-learn tqdm torch torchvision

In [ ]:
import os, json, numpy as np, nibabel as nib
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from tqdm import tqdm

PROJECT_ROOT = os.path.expanduser("~/Documents/GitHub/fyp-3d-ct")
DATA_ROOT    = os.path.join(PROJECT_ROOT, "data")
VOLUMES_DIR  = os.path.join(DATA_ROOT, "data_volumes")
REXCT_DIR    = os.path.join(DATA_ROOT, "rexgrounding-ct")
DATASET_JSON = os.path.join(REXCT_DIR, "dataset.json")

# ── Training config ───────────────────────────────────────────
CONFIG = {
    "batch_size"  : 2,       # keep low — CT volumes are huge
    "epochs"      : 20,
    "lr"          : 1e-4,
    "val_split"   : 0.1,     # 10% for validation
    "num_workers" : 0,       # 0 = safe on Mac, increase on Linux/GPU
    "checkpoint_dir": os.path.join(PROJECT_ROOT, "models/merlin/checkpoints"),
    "results_dir"   : os.path.join(PROJECT_ROOT, "models/merlin/results"),
    "device"        : "cuda" if torch.cuda.is_available() else
                      "mps"  if torch.backends.mps.is_available() else "cpu"
}

os.makedirs(CONFIG["checkpoint_dir"], exist_ok=True)
os.makedirs(CONFIG["results_dir"],    exist_ok=True)

print(f"Device : {CONFIG['device']}")
print(f"Config : {CONFIG}")

In [ ]:
# Map free-text findings → class index via Claude API classification
LABEL2IDX = {
    "Lung Nodule" : 0,
    "Lung opacity": 1,
    "Consolidation": 2,
    "Atelectasis"  : 3,
    "Others"       : 4
}
IDX2LABEL = {v: k for k, v in LABEL2IDX.items()}
NUM_CLASSES = len(LABEL2IDX)
print(f"Classes: {LABEL2IDX}")

In [ ]:
# Map free-text findings → class index via Claude API classification
LABEL2IDX = {
    "Lung Nodule" : 0,
    "Lung opacity": 1,
    "Consolidation": 2,
    "Atelectasis"  : 3,
    "Others"       : 4
}
IDX2LABEL = {v: k for k, v in LABEL2IDX.items()}
NUM_CLASSES = len(LABEL2IDX)
print(f"Classes: {LABEL2IDX}")

In [ ]:
import urllib.request

def classify_finding(finding_text: str) -> str:
    VALID = list(LABEL2IDX.keys())
    prompt = f"""Analyze the provided input and identify the lung condition. You must strictly limit your output to one of the following exact categories: 'Lung Nodule', 'Lung opacity', 'Consolidation', or 'Atelectasis'. If any other disease, abnormality, or condition is detected that is not on this list, classify it strictly as 'Others'. Output only the category name, without any extra text or explanations.

Input: {finding_text}"""

    payload = json.dumps({
        "model"     : "claude-sonnet-4-20250514",
        "max_tokens": 20,
        "messages"  : [{"role": "user", "content": prompt}]
    }).encode("utf-8")

    req = urllib.request.Request(
        "https://api.anthropic.com/v1/messages",
        data=payload,
        headers={"Content-Type": "application/json",
                 "anthropic-version": "2023-06-01"},
        method="POST"
    )
    try:
        with urllib.request.urlopen(req) as resp:
            data     = json.loads(resp.read().decode())
            category = data["content"][0]["text"].strip()
            return category if category in VALID else "Others"
    except:
        return "Others"

In [ ]:
LABELS_CACHE = os.path.join(REXCT_DIR, "labels_cache.json")

def build_labels_cache(samples, ct_index):
    """Classify every finding once and save to disk."""
    if os.path.exists(LABELS_CACHE):
        print(f"Labels cache found — loading from {LABELS_CACHE}")
        with open(LABELS_CACHE) as f:
            return json.load(f)

    print(f"Building labels cache for {len(samples)} samples...")
    cache = {}
    for i, s in enumerate(tqdm(samples)):
        name = s["name"]
        if name not in ct_index:
            continue  # skip missing files
        
        # Take the first finding as the primary label
        findings = s.get("findings", {})
        if not findings:
            label = "Others"
        else:
            first_text = list(findings.values())[0]
            label      = classify_finding(first_text)
        
        cache[name] = {
            "label"    : label,
            "label_idx": LABEL2IDX[label],
            "findings" : findings
        }
        
        # Save every 50 samples in case of interruption
        if (i + 1) % 50 == 0:
            with open(LABELS_CACHE, "w") as f:
                json.dump(cache, f, indent=2)
            print(f"  Saved cache at {i+1} samples")

    with open(LABELS_CACHE, "w") as f:
        json.dump(cache, f, indent=2)
    
    print(f"Cache complete — {len(cache)} samples labeled")
    return cache


# Index files then build cache
ct_index = {}
for root, dirs, files in os.walk(VOLUMES_DIR):
    for f in files:
        if f.endswith(".nii.gz"):
            ct_index[f] = os.path.join(root, f)

with open(DATASET_JSON) as f:
    metadata = json.load(f)
samples = metadata if isinstance(metadata, list) else list(metadata.values())[0]
matched = [s for s in samples if s["name"] in ct_index]

print(f"Total in JSON : {len(samples)}")
print(f"On disk       : {len(ct_index)}")
print(f"Matched       : {len(matched)}")

labels_cache = build_labels_cache(matched, ct_index)

In [ ]:
class RexGroundingDataset(Dataset):
    def __init__(self, samples, ct_index, labels_cache):
        # Only keep samples that have a file AND a label
        self.samples      = [s for s in samples
                             if s["name"] in ct_index
                             and s["name"] in labels_cache]
        self.ct_index     = ct_index
        self.labels_cache = labels_cache
        print(f"Dataset ready: {len(self.samples)} samples")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample   = self.samples[idx]
        name     = sample["name"]
        ct_path  = self.ct_index[name]
        label_idx = self.labels_cache[name]["label_idx"]

        # Load & preprocess CT
        img    = nib.load(ct_path)
        volume = img.get_fdata()
        volume = np.clip(volume, -1000, 400)
        volume = (volume + 1000) / 1400.0
        volume = np.nan_to_num(volume, nan=0.0)

        # Resize to fixed size so batching works (Merlin expects consistent input)
        volume = self._resize(volume, target=(128, 128, 64))

        tensor = torch.tensor(volume, dtype=torch.float32).unsqueeze(0)  # (1,H,W,D)
        return tensor, torch.tensor(label_idx, dtype=torch.long), name

    def _resize(self, volume, target=(128, 128, 64)):
        """Simple resize by slicing/padding to target shape."""
        th, tw, td = target
        h, w, d    = volume.shape

        # Crop or pad each dimension
        def fit(arr, size, axis):
            cur = arr.shape[axis]
            if cur >= size:
                slices = [slice(None)] * arr.ndim
                slices[axis] = slice(0, size)
                return arr[tuple(slices)]
            else:
                pad = [(0, 0)] * arr.ndim
                pad[axis] = (0, size - cur)
                return np.pad(arr, pad, mode="constant", constant_values=0)

        volume = fit(volume, th, 0)
        volume = fit(volume, tw, 1)
        volume = fit(volume, td, 2)
        return volume

In [ ]:
from merlin import Merlin

class MerlinClassifier(nn.Module):
    def __init__(self, num_classes=5):
        super().__init__()
        self.backbone = Merlin(ImageEmbedding=True)
        
        # Get embedding dimension by doing one dummy forward pass
        with torch.no_grad():
            dummy = torch.zeros(1, 1, 128, 128, 64)
            emb   = self.backbone(dummy)
            emb_dim = emb.shape[-1]
        
        print(f"Embedding dim: {emb_dim}")
        
        # Classification head on top of Merlin embedding
        self.classifier = nn.Sequential(
            nn.Linear(emb_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        embedding = self.backbone(x)           # (B, emb_dim)
        logits    = self.classifier(embedding) # (B, num_classes)
        return logits, embedding


# Build model
model     = MerlinClassifier(num_classes=NUM_CLASSES).to(CONFIG["device"])
optimizer = torch.optim.Adam(model.parameters(), lr=CONFIG["lr"])
criterion = nn.CrossEntropyLoss()
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)

print("Model ready for training")

In [ ]:
from torch.utils.data import DataLoader

# Split matched samples
train_samples, val_samples = train_test_split(
    matched, test_size=CONFIG["val_split"], random_state=42
)
print(f"Train: {len(train_samples)}  |  Val: {len(val_samples)}")

train_dataset = RexGroundingDataset(train_samples, ct_index, labels_cache)
val_dataset   = RexGroundingDataset(val_samples,   ct_index, labels_cache)

train_loader = DataLoader(train_dataset,
                          batch_size=CONFIG["batch_size"],
                          shuffle=True,
                          num_workers=CONFIG["num_workers"])

val_loader   = DataLoader(val_dataset,
                          batch_size=CONFIG["batch_size"],
                          shuffle=False,
                          num_workers=CONFIG["num_workers"])

print(f"Train batches: {len(train_loader)}  Val batches: {len(val_loader)}")

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss, correct, total = 0, 0, 0

    for ct, labels, _ in tqdm(loader, desc="  Train", leave=False):
        ct, labels = ct.to(device), labels.to(device)

        optimizer.zero_grad()
        logits, _ = model(ct)
        loss      = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        preds       = logits.argmax(dim=1)
        correct    += (preds == labels).sum().item()
        total      += labels.size(0)

    return total_loss / len(loader), correct / total


def validate(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    all_preds, all_labels = [], []

    with torch.no_grad():
        for ct, labels, _ in tqdm(loader, desc="  Val  ", leave=False):
            ct, labels = ct.to(device), labels.to(device)
            logits, _  = model(ct)
            loss        = criterion(logits, labels)

            total_loss += loss.item()
            preds       = logits.argmax(dim=1)
            correct    += (preds == labels).sum().item()
            total      += labels.size(0)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    return total_loss / len(loader), correct / total, all_preds, all_labels

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

# Save history
with open(os.path.join(CONFIG["results_dir"], "history.json"), "w") as f:
    json.dump(history, f, indent=2)

# Plot training curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor("#0e0e0e")

for ax in (ax1, ax2):
    ax.set_facecolor("#1a1a1a")
    ax.tick_params(colors="white")
    for spine in ax.spines.values():
        spine.set_edgecolor("white")

epochs = range(1, CONFIG["epochs"] + 1)
ax1.plot(epochs, history["train_loss"], "cyan",  label="Train loss")
ax1.plot(epochs, history["val_loss"],   "orange",label="Val loss")
ax1.set_title("Loss", color="white"); ax1.legend(); ax1.set_xlabel("Epoch", color="white")

ax2.plot(epochs, history["train_acc"], "cyan",  label="Train acc")
ax2.plot(epochs, history["val_acc"],   "orange",label="Val acc")
ax2.set_title("Accuracy", color="white"); ax2.legend(); ax2.set_xlabel("Epoch", color="white")

plt.tight_layout()
plt.savefig(os.path.join(CONFIG["results_dir"], "training_curves.png"),
            dpi=150, bbox_inches="tight", facecolor=fig.get_facecolor())
plt.show()

# Classification report
print("\nClassification Report:")
print(classification_report(val_labels, val_preds,
      target_names=list(LABEL2IDX.keys())))

# Confusion matrix
cm  = confusion_matrix(val_labels, val_preds)
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=list(LABEL2IDX.keys()),
            yticklabels=list(LABEL2IDX.keys()), ax=ax)
ax.set_title("Confusion Matrix")
ax.set_xlabel("Predicted"); ax.set_ylabel("True")
plt.tight_layout()
plt.savefig(os.path.join(CONFIG["results_dir"], "confusion_matrix.png"), dpi=150)
plt.show()